# Notebook 2 (V9) — Absorbing-Regime Balanced-Growth Path

When the regime $z = b$ (absorbing), the equilibrium is characterised by a stationary normalised solution (V9 Assumption `ass_primitive_bg_branch`). With constant labour shares $\bar\varphi_b, \bar\varphi_W$:

$$
N_{US,t+1} = G_{N,US} N_{US,t}, \quad N_{W,t+1} = G_{N,W} N_{W,t},
$$
$$
Y_{US,t} \propto N_{US,t}^{\nu_b}, \quad q_{US,t} \propto N_{US,t}^{\nu_b - 1}, \quad d_{US,t} \propto N_{US,t}^{\nu_b-1},
$$
and similarly for RoW with exponent $\xi_W$.

**The 7×7 BGP system** (Section 6 of `claude_code_prompt_v9_numerical_solver.md`):
Unknowns $\bar y_b = (\bar\varphi_{US}, \bar\varphi_W, \bar\omega, \bar\theta, \bar\omega^*, \bar R_f, \bar R_f^W)$. Bond clearing pins $\bar\theta_{US}^* = -\bar\theta \bar A / \bar A^*$ and $\bar\theta_W^* = 0$.

1. US stock-market clearing
2. RoW stock-market clearing
3. US $\omega$ FOC
4. US $\theta$ FOC
5. RoW $\omega^*$ FOC
6. RoW $R_f^W$ FOC (deterministic ⇒ $R_f^W = R_p^*$)
7. RoW $\theta_{US}^*$ FOC (with convenience-yield wedge $\chi/\theta_{US}^*$)

In [ ]:
using Pkg; Pkg.activate(".")
include("TwoCountryProductionOLG.jl")
using Plots, LaTeXStrings, Printf
gr()

In [ ]:
p = ProductionParams(common_world_growth=true)
validate_params(p)
println("Parameters validated.")

## 1. BGP at the Initial State $(N_{US,0}, N_{W,0})$

In [ ]:
bgp = solve_bgp_at(p, p.N_US_0, p.N_W_0)

println("═══ BGP at initial state ($(p.N_US_0), $(p.N_W_0)) ═══")
@printf("  φ_US = %.4f, φ_W = %.4f\n", bgp.φ_US, bgp.φ_W)
@printf("  ω = %.4f, ω* = %.4f\n", bgp.ω, bgp.ω_star)
@printf("  θ = %+.4f, θ_US^* = %+.4f\n", bgp.θ, bgp.θ_US_star)
@printf("  R_f = %.4f, R_f^W = %.4f\n", bgp.R_f, bgp.R_f_W)
@printf("  R_US = %.4f, R_W = %.4f, R_p = %.4f\n", bgp.R_US, bgp.R_W, bgp.R_p)
@printf("  R_A = %.4f, R_A^* = %.4f\n", bgp.R_A, bgp.R_A_star)
@printf("  G_N_US = %.4f, G_N_W = %.4f\n", bgp.G_N_US, bgp.G_N_W)
@printf("  G_b = %.4f, G_W = %.4f (gap = %+.2e)\n",
        bgp.G_N_US^p.ν_b, bgp.G_N_W^p.ξ_W,
        bgp.G_N_US^p.ν_b - bgp.G_N_W^p.ξ_W)
@printf("  Q_US = %.4f, Q_W = %.4f, e_US = %.4f, e_W = %.4f\n",
        bgp.Q_US, bgp.Q_W, bgp.e_US, bgp.e_W)
@printf("  Mkt-cap identity gap: %+.2e\n",
        bgp.Q_US + bgp.Q_W - (p.β*bgp.e_US + (p.β+p.χ)/(1+p.χ)*bgp.e_W))
@printf("  Converged: %s, ‖F‖ = %.2e\n", bgp.converged, bgp.residual_norm)

## 2. Common-World-Growth Calibration

If `common_world_growth=true`, calibrate $\nu_b$ to satisfy $G_b = G_W$. This makes the BGP fully stationary (no drift in relative country size after a switch).

In [ ]:
cal = calibrate_common_growth(p, p.N_US_0, p.N_W_0; verbose=false)
p_cal = cal.params
bgp_cal = cal.bgp

@printf("Calibrated ν_b = %.6f (was %.6f)\n", p_cal.ν_b, p.ν_b)
@printf("Common-growth gap: G_b - G_W = %+.2e\n",
        bgp_cal.G_N_US^p_cal.ν_b - bgp_cal.G_N_W^p_cal.ξ_W)

## 3. BGP Stationarity Across Different $(N_{US}, N_W)$

In [ ]:
states = [(1.0, 1.0), (5.0, 5.0), (10.0, 10.0), (50.0, 50.0), (100.0, 100.0)]
println("BGP at multiple equal-N states (under common-growth calibration):")
println("  N_US    N_W     φ_US    φ_W     ω      θ        ω*     R_f    R_A")
for (N_US, N_W) in states
    b = solve_bgp_at(p_cal, N_US, N_W)
    @printf("  %5.1f  %5.1f   %.4f  %.4f  %.4f  %+.4f  %.4f  %.4f  %.4f\n",
            N_US, N_W, b.φ_US, b.φ_W, b.ω, b.θ, b.ω_star, b.R_f, b.R_A)
end

## 4. Sensitivity to Relative Country Size $(N_W/N_{US})$

In [ ]:
ratios = [0.5, 1.0, 2.0, 5.0, 10.0]
results = [solve_bgp_at(p_cal, 1.0, r) for r in ratios]

p1 = plot(ratios, [r.φ_US for r in results], lw=2, marker=:circle, label=L"\bar\varphi_{US}",
          xlabel=L"N_W/N_{US}", ylabel=L"\bar\varphi", xscale=:log10,
          title="BGP labour allocations")
plot!(p1, ratios, [r.φ_W for r in results], lw=2, marker=:square, label=L"\bar\varphi_W")

p2 = plot(ratios, [r.ω for r in results], lw=2, marker=:circle, label=L"\bar\omega",
          xlabel=L"N_W/N_{US}", ylabel="weight", xscale=:log10,
          title="BGP portfolio weights")
plot!(p2, ratios, [r.ω_star for r in results], lw=2, marker=:square, label=L"\bar\omega^*")

p3 = plot(ratios, [r.θ for r in results], lw=2, marker=:circle, label=L"\bar\theta",
          xlabel=L"N_W/N_{US}", ylabel="bond share", xscale=:log10,
          title="BGP bond shares")
plot!(p3, ratios, [r.θ_US_star for r in results], lw=2, marker=:square, label=L"\bar\theta_{US}^*")

p4 = plot(ratios, [r.R_f for r in results], lw=2, marker=:circle, label=L"R_f",
          xlabel=L"N_W/N_{US}", ylabel="return", xscale=:log10,
          title="BGP risk-free rates")
plot!(p4, ratios, [r.R_f_W for r in results], lw=2, marker=:square, label=L"R_f^W")
plot!(p4, ratios, [r.R_A for r in results], lw=2, marker=:utriangle, label=L"R_A")

plot(p1, p2, p3, p4, layout=(2,2), size=(900, 700))

## 5. Per-Variety Prices and Aggregate Market Cap along BGP

By scaling laws (V9 Prop. `prop_balanced_growth_primitives`):
$$
q_{US,t} = \bar q_{US} N_{US,t}^{\nu_b - 1}, \qquad \mathcal{Q}_{US,t} = \bar{\mathcal{Q}}_{US} N_{US,t}^{\nu_b}.
$$

In [ ]:
Ns = 10 .^ range(0, 4, length=120)
qs = Float64[]; ds = Float64[]; Qs = Float64[]; Rs = Float64[]
for N in Ns
    b = solve_bgp_at(p_cal, N, N)
    push!(qs, b.q_US); push!(ds, b.d_US); push!(Qs, b.Q_US); push!(Rs, b.R_US)
end

p1 = plot(Ns, qs, lw=2, label=L"q_{US}^b", xscale=:log10, yscale=:log10,
          xlabel=L"N_{US}=N_W", ylabel="per-variety price (log)",
          title="Per-variety stock price along BGP")
plot!(p1, Ns, ds, lw=2, label=L"d_{US}^b", ls=:dash)

p2 = plot(Ns, Qs, lw=2, label=L"\mathcal{Q}_{US}^b", xscale=:log10, yscale=:log10,
          xlabel=L"N_{US}=N_W", ylabel="aggregate market cap (log)",
          title="Aggregate stock-market capitalisation")

p3 = plot(Ns, Rs, lw=2, label=L"R_{US}^b", xscale=:log10,
          xlabel=L"N_{US}=N_W", ylabel="return",
          title="Per-variety return on absorbing BGP")
hline!(p3, [1.0], ls=:dash, color=:black, label="")

plot(p1, p2, p3, layout=(1,3), size=(1200, 380))

## 6. Verifying the Aggregate Market-Cap Identity

In [ ]:
println("V9 Aggregate market-cap identity:")
println("   N_US    Q_US+Q_W    βe_US+(β+χ)/(1+χ)e_W    gap")
for N in [1.0, 5.0, 10.0, 50.0, 100.0]
    b = solve_bgp_at(p_cal, N, N)
    LHS = b.Q_US + b.Q_W
    RHS = p.β*b.e_US + (p.β+p.χ)/(1+p.χ)*b.e_W
    @printf("  %5.1f   %.4f       %.4f                %+.2e\n", N, LHS, RHS, LHS-RHS)
end

## Summary

- The BGP solver pins down $\bar\varphi_b, \bar\varphi_W$ together with portfolio weights $\omega, \omega^*$, bond share $\theta$, and risk-free rates $R_f, R_f^W$.
- Common-world-growth calibration adjusts $\nu_b$ via damped fixed-point iteration so that $G_b = G_W$.
- Once calibrated, the BGP is approximately stationary across $(N_{US}, N_W)$ levels.
- The aggregate market-cap identity holds exactly at every BGP point.

Next notebook: solve the **unbalanced branch** by forward-backward iteration.